# ** Create Your First Delta Table**

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.test")


DataFrame[]

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.test.orders (
 order_id INT,
 customer STRING,
 amount DOUBLE
)
""")

DataFrame[]

In [0]:
spark.sql("""
INSERT INTO workspace.test.orders VALUES
 (1, 'Amir', 120.0),
 (2, 'Priya', 75.5),
 (3, 'Jon', 200.0)
""")


DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(spark.sql("SELECT * FROM workspace.test.orders ORDER BY order_id"))


order_id,customer,amount
1,Amir,120.0
2,Priya,75.5
3,Jon,200.0


In [0]:
%sql
DESCRIBE HISTORY workspace.test.orders

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-07-13T10:05:49.000Z,315897387088307,codebakery8889@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(575048764221192),018c03c4-4df8-4286-bb04-9826bcf61585,0713-095838-ciiti7lz-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 3299, p25FileSize -> 2102, numDeletionVectorsRemoved -> 1, minFileSize -> 2102, numAddedFiles -> 1, maxFileSize -> 2102, p75FileSize -> 2102, p50FileSize -> 2102, numAddedBytes -> 2102)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
2,2026-07-13T10:05:46.000Z,315897387088307,codebakery8889@gmail.com,UPDATE,"Map(predicate -> [""(customer#11742 = Jon)""])",null,List(575048764221192),018c03c4-4df8-4286-bb04-9826bcf61585,0713-095838-ciiti7lz-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 5417, numDeletionVectorsUpdated -> 0, scanTimeMs -> 2753, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1999, rewriteTimeMs -> 2567)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-07-13T10:02:55.000Z,315897387088307,codebakery8889@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(575048764221192),dadc44ec-a627-4022-9402-857f35a815ef,0713-095838-ciiti7lz-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 1300)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-07-13T09:59:55.000Z,315897387088307,codebakery8889@gmail.com,CREATE OR REPLACE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-c45c96e5-1908-4544-b53c-f041b21acc98"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-471dd766-7c45-431b-9b99-e16a38b5ad48""}, statsOnLoad -> false)",null,List(575048764221192),b7e15e86-17a9-4428-934b-10c0307197da,0713-095838-ciiti7lz-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


In [0]:
spark.sql("UPDATE workspace.test.orders SET amount = amount * 1.1 WHERE customer = 'Jon'")

DataFrame[num_affected_rows: bigint]

In [0]:
%sql
select * from workspace.test.orders

order_id,customer,amount
1,Amir,120.0
2,Priya,75.5
3,Jon,220.00000000000003


In [0]:
display(spark.sql("DESCRIBE HISTORY workspace.test.orders").select("version",
"operation", "operationParameters"))


version,operation,operationParameters
5,RESTORE,"Map(version -> 1, timestamp -> null)"
4,RESTORE,"Map(version -> 2, timestamp -> null)"
3,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)"
2,UPDATE,"Map(predicate -> [""(customer#11742 = Jon)""])"
1,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])"
0,CREATE OR REPLACE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-c45c96e5-1908-4544-b53c-f041b21acc98"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-471dd766-7c45-431b-9b99-e16a38b5ad48""}, statsOnLoad -> false)"


In [0]:
%sql
RESTORE TABLE workspace.test.orders TO VERSION AS OF 3;

table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
2102,1,1,1,1300,2102


In [0]:
try:
 spark.sql("INSERT INTO workspace.test.orders VALUES (4, 'BadRow', 'not_a_number')")
except Exception as e:
 print("Write failed as expected:", type(e).__name__)
print("Row count after the failed write:", spark.table("workspace.default.orders").count())

Write failed as expected: NumberFormatException
Row count after the failed write: 3


In [0]:
%sql
VACUUM workspace.test.orders RETAIN 1 HOURS;

path
""


In [0]:
%sql
SET spark.databricks.delta.retentionDurationCheck.enabled = false;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6830919549535028>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'SET spark.databricks.delta.retentionDurationCheck.enabled = false;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:213, in SqlMagic.sql(self, line, cell)
    206 except BaseException as e:
    207    

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.test.events (
 event_id INT,
 event_type STRING
)
""")
from pyspark.sql import Row
new_events = spark.createDataFrame([
 Row(event_id=1, event_type='click', region='APAC'),
 Row(event_id=2, event_type='purchase', region='EU'),
])
try:
 new_events.write.format("delta").mode("overwrite").saveAsTable("workspace.test.events")
except Exception as e:
 print("Blocked as expected:", type(e).__name__)

Blocked as expected: AnalysisException


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.test.events (
    event_id INT,
    event_type STRING
)
""")

DataFrame[]

In [0]:
from pyspark.sql import Row

new_events = spark.createDataFrame([
    Row(event_id=1, event_type='click', region='APAC'),
    Row(event_id=2, event_type='purchase', region='EU'),
])

In [0]:
new_events.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.test.events")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6830919549535032>, line 4
      1 new_events.write \
      2     .format("delta") \
      3     .mode("overwrite") \
----> 4     .saveAsTable("workspace.test.events")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1536     req.user_context.user_id = self

In [0]:
spark.table("workspace.test.events").printSchema()

root
 |-- event_id: integer (nullable = true)
 |-- event_type: string (nullable = true)



In [0]:
new_events.printSchema()

root
 |-- event_id: integer (nullable = true)
 |-- event_type: string (nullable = true)
 |-- region: string (nullable = true)



In [0]:
new_events.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.test.events")

In [0]:
%sql
select * from workspace.test.events

event_id,event_type,region
1,click,APAC
2,purchase,EU


In [0]:
%sql
DESCRIBE HISTORY workspace.test.events

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-07-13T10:25:21.000Z,315897387088307,codebakery8889@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-0ad266e6-c08e-4884-9cb0-f655de8b3b7a"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-5770f49a-18b7-4ab9-bc2e-883ab0121748""}, statsOnLoad -> true)",null,List(575048764221192),a7c6a461-d04e-4694-ba21-8f2a1f48cae1,0713-095838-ciiti7lz-v2n,2,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 2, numOutputBytes -> 1305)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
2,2026-07-13T10:22:30.000Z,315897387088307,codebakery8889@gmail.com,CREATE OR REPLACE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-0ad266e6-c08e-4884-9cb0-f655de8b3b7a"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-5770f49a-18b7-4ab9-bc2e-883ab0121748""}, statsOnLoad -> false)",null,List(575048764221192),b13216a1-7cbf-4433-88c9-4544560485ff,0713-095838-ciiti7lz-v2n,1,WriteSerializable,false,Map(),null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-07-13T10:21:01.000Z,315897387088307,codebakery8889@gmail.com,CREATE OR REPLACE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-0ad266e6-c08e-4884-9cb0-f655de8b3b7a"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-5770f49a-18b7-4ab9-bc2e-883ab0121748""}, statsOnLoad -> false)",null,List(575048764221192),dd94e352-64e0-4696-b11b-d66daae5202f,0713-095838-ciiti7lz-v2n,0,WriteSerializable,false,Map(),null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-07-13T10:20:47.000Z,315897387088307,codebakery8889@gmail.com,CREATE OR REPLACE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-0ad266e6-c08e-4884-9cb0-f655de8b3b7a"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-5770f49a-18b7-4ab9-bc2e-883ab0121748""}, statsOnLoad -> false)",null,List(575048764221192),fb1fa06f-4137-49e6-ad94-2aebea5275a5,0713-095838-ciiti7lz-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
